# 범죄 피해 지원 길잡이 — 데이터 분석 초안
1. 전국·서울 범죄 현황을 그래프와 지도로 보기 (pandas, plotly, folium)
2. 겪은 일을 설명하면 받을 수 있는 지원제도를 안내하는 AI (OpenAI)

**먼저 할 일:** `data/seoul_gu.csv`, `data/sido.csv` 의 **인구** 칸을 채우세요. (행정안전부 주민등록 인구통계, 범죄 통계와 같은 2025년 12월 기준)
인구 칸이 비어 있으면 '발생 건수' 지도로 그려집니다. 지도는 구역 전체를 색으로 칠하는 방식이며, 구역 경계 파일은 처음 실행할 때 인터넷에서 내려받습니다.

## 1. 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import folium

In [ ]:
# 지역별 범죄 통계(시군구별), 장소별 범죄 통계
crime = pd.read_csv('data/경찰청_범죄 발생 지역별 통계_20251231.csv', encoding='cp949')
place = pd.read_csv('data/경찰청_범죄 발생 장소별 통계_20251231.csv', encoding='cp949')
crime.head()

In [ ]:
# 범죄 종류(범죄대분류)를 확인한다.
crime['범죄대분류'].unique()

In [ ]:
# 어떤 범죄를 볼지 고른다. ('전체', '강력범죄', '폭력범죄', '절도범죄', '지능범죄' 등으로 바꿔서 다시 실행해 보세요)
crime_type = '전체'

if crime_type == '전체':
    data = crime
    place_data = place
else:
    data = crime[crime['범죄대분류'] == crime_type]
    place_data = place[place['범죄대분류'] == crime_type]

## 2. 어떤 범죄가 많을까? (전국)

In [ ]:
# 외국 컬럼을 뺀 지역 컬럼 목록 (컬럼 이름 앞 두 칸은 범죄 분류)
region_cols = [c for c in data.columns[2:] if not c.startswith('외국')]

# 범죄 종류별 전국 건수 (가로로 합계)
type_sum = data.copy()
type_sum['전국'] = type_sum[region_cols].sum(axis=1)
type_sum = type_sum.sort_values('전국', ascending=False).head(10)

px.bar(data_frame=type_sum, x='범죄중분류', y='전국', color='범죄중분류')

In [ ]:
# 범죄가 일어난 장소 TOP 10 (장소별 통계)
place_cols = place_data.columns[2:]
place_sum = place_data[place_cols].sum().reset_index()
place_sum.columns = ['장소', '건수']
place_sum = place_sum.sort_values('건수', ascending=False).head(10)

px.bar(data_frame=place_sum, x='장소', y='건수', color='장소')

## 3. 지도 그리기 준비

In [ ]:
# 인구가 있으면 '인구 10만 명당 건수', 없으면 '건수'를 값으로 사용하고
# 값에 따라 등급(높음/보통/낮음)을 정한다.
def prepare(df):
    df = df.copy()
    if df['인구'].isna().any():
        df['값'] = df['건수']
    else:
        df['값'] = df['건수'] / df['인구'] * 100000
    q1 = df['값'].quantile(0.33)
    q2 = df['값'].quantile(0.66)
    df['등급'] = np.where(df['값'] >= q2, '높음', np.where(df['값'] >= q1, '보통', '낮음'))
    return df

In [ ]:
import json
import os
import urllib.request

# 배경 지도 설정
# 수업 11장의 xyzservices 키가 있으면 아래에 붙여넣으세요 (깔끔한 밝은 배경 지도가 나옵니다).
# 비워두면 배경 지도 없이 구역만 그립니다. (GitHub에 올릴 때는 키를 꼭 지우세요)
CARTO_KEY = ''

if CARTO_KEY == '':
    base_tiles = None
else:
    import xyzservices.providers as xyz
    base_tiles = xyz.CartoDB.Positron(apikey=CARTO_KEY)

# 지도에 칠할 구역(경계) 파일을 내려받는 함수. 한 번 받으면 data 폴더에 저장해 두고 다시 쓴다.
def load_geo(path, urls):
    if not os.path.exists(path):
        for url in urls:
            try:
                urllib.request.urlretrieve(url, path)
                print('내려받음:', url)
                break
            except Exception as e:
                print('실패:', url, e)
    if not os.path.exists(path):
        raise FileNotFoundError(f'경계 파일을 받지 못했습니다. README의 안내대로 {path} 를 직접 저장하세요.')
    return json.load(open(path, encoding='utf-8'))

# 값이 클수록 진하게 구역 전체를 칠하는 지도(단계 구분도)를 만드는 함수
def make_choropleth(geo, df, key_col, value_col, center, zoom, legend):
    # 값 구간을 5등분: 한두 지역이 너무 커도 색이 고르게 나뉘도록 분위수로 나눈다.
    bins = sorted(set(df[value_col].quantile([0, 0.2, 0.4, 0.6, 0.8, 1]).tolist()))
    m = folium.Map(location=center, zoom_start=zoom, tiles=base_tiles)
    folium.Choropleth(
        geo_data=geo,
        data=df,
        columns=[key_col, value_col],
        key_on='feature.properties.name',
        bins=bins,
        fill_color='YlOrRd',
        fill_opacity=0.8,
        line_color='white',
        line_weight=1,
        line_opacity=1,
        nan_fill_color='lightgray',
        legend_name=legend
    ).add_to(m)
    # 마우스를 올리면 지역 이름과 값이 보이도록 한다.
    value_dict = dict(zip(df[key_col], df[value_col]))
    for f in geo['features']:
        f['properties']['값'] = f"{value_dict.get(f['properties']['name'], 0):,.0f}"
    folium.GeoJson(
        geo,
        style_function=lambda x: {'fillOpacity': 0, 'weight': 0},
        tooltip=folium.GeoJsonTooltip(fields=['name', '값'], aliases=['지역', '값'])
    ).add_to(m)
    return m

## 4. 서울 자치구 범죄 지도

In [ ]:
# 서울 자치구 컬럼만 골라 구별 건수를 구한다.
seoul_cols = [c for c in data.columns if c.startswith('서울')]
seoul = data[seoul_cols].sum().reset_index()
seoul.columns = ['지역', '건수']
seoul['구'] = seoul['지역'].str.replace('서울 ', '')

# 인구 합치기
gu = pd.read_csv('data/seoul_gu.csv', encoding='cp949')
seoul = seoul.merge(gu, on='구')
seoul = prepare(seoul)

if seoul['인구'].isna().any():
    unit = '발생 건수'
else:
    unit = '인구 10만 명당 발생 건수'

seoul.sort_values('값', ascending=False).head()

In [ ]:
# 서울 구별 값 막대 그래프
px.bar(data_frame=seoul.sort_values('값', ascending=False), x='구', y='값', color='등급')

In [ ]:
# 서울 구역 경계 파일 (처음 한 번 내려받습니다)
seoul_geo = load_geo('data/seoul_geo.json', [
    'https://raw.githubusercontent.com/southkorea/seoul-maps/master/kostat/2013/json/seoul_municipalities_geo_simple.json',
    'https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2013/json/skorea_municipalities_geo_simple.json',
])

# 서울(코드가 11로 시작)만 남긴다.
seoul_geo['features'] = [f for f in seoul_geo['features'] if str(f['properties']['code']).startswith('11')]

# 경계 파일 확인: 구 개수(25개), 속성, 통계와 이름이 안 맞는 구
print(len(seoul_geo['features']), seoul_geo['features'][0]['properties'])
geo_names = [f['properties']['name'] for f in seoul_geo['features']]
print('지도에 없는 구:', [g for g in seoul['구'] if g not in geo_names])

In [ ]:
# 서울 지도: 값이 클수록 진한 색 / 구를 가리키면 이름과 값이 보입니다.
seoul_map = make_choropleth(seoul_geo, seoul, '구', '값', [37.55, 126.98], 11, unit)
#seoul_map.save('seoul_crime_map.html')
seoul_map

## 5. 전국(시도) 범죄 지도

In [ ]:
# 지역 이름의 첫 단어(서울, 경기도, ...)로 시도를 만든다.
total = data[region_cols].sum().reset_index()
total.columns = ['지역', '건수']
total['시도'] = [x.split()[0] for x in total['지역']]

sido = total.groupby('시도', as_index=False).agg(건수=('건수', 'sum'))
sido_info = pd.read_csv('data/sido.csv', encoding='cp949')
sido = sido.merge(sido_info, on='시도')
sido = prepare(sido)

if sido['인구'].isna().any():
    unit = '발생 건수'
else:
    unit = '인구 10만 명당 발생 건수'

sido.sort_values('값', ascending=False).head()

In [ ]:
# 시도 경계 파일 (처음 한 번 내려받습니다)
korea_geo = load_geo('data/sido_geo.json', [
    'https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2013/json/skorea_provinces_geo_simple.json',
])

# 경계 파일 확인: 통계와 이름이 안 맞는 시도
geo_names = [f['properties']['name'] for f in korea_geo['features']]
print(len(geo_names), '개 /', '지도에 없는 시도:', [g for g in sido['지도이름'] if g not in geo_names])

In [ ]:
korea_map = make_choropleth(korea_geo, sido, '지도이름', '값', [36.3, 127.8], 7, unit)
#korea_map.save('korea_crime_map.html')
korea_map

## 6. 겪은 일을 설명하면 지원제도를 안내하는 AI
수업 때 배운 방식 그대로, 질문(prompt)에 **지원제도 목록**을 넣어서 AI가 목록 안에서만 안내하도록 합니다.
(주의: 이 내용은 외부 AI 서비스로 전송됩니다. 실제 피해자의 개인정보는 넣지 마세요.)

In [ ]:
from openai import OpenAI

# AI 서비스와 통신할 객체 생성 (환경변수 OPENAI_API_KEY 필요)
client = OpenAI()
MODEL = 'gpt-5.6-luna'    # 수업 때 사용한 모델 이름. 사용할 수 있는 모델로 바꿔도 됩니다.

# 안내할 수 있는 지원제도 목록
supports = pd.read_csv('data/supports.csv', encoding='cp949')
supports

In [ ]:
# 피해 상황 (예시는 가상의 상황입니다)
story = "모르는 번호로 온 문자의 링크를 눌렀는데, 조금 뒤 제 계좌에서 돈이 빠져나갔어요."

prompt = f"""
당신은 범죄 피해자에게 받을 수 있는 지원제도를 안내하는 도우미입니다.

[피해자가 설명한 상황]
{story}

[안내할 수 있는 지원제도 목록]
{supports.to_string()}

다음 규칙을 지켜 안내해주세요.
1. 위 목록에 있는 제도만 안내하고, 목록에 없는 제도·금액·기한은 지어내지 마세요.
2. 먼저 어떤 유형의 피해일 수 있는지 한두 문장으로 설명하세요. 확정하지 말고 '~일 수 있습니다'라고 쓰세요.
3. 신체 위험이 있거나 급박해 보이면 가장 먼저 112 신고를 안내하세요.
4. 해당할 수 있는 제도를 '제도명 - 하는 일 - 연락처' 순서로 정리하세요.
5. 확인상태가 '확인필요'인 제도는 공식 기관에 다시 확인하라고 덧붙이세요.
6. 마지막에 '법률 자문이 아닌 참고용 안내입니다'라고 쓰세요.
"""

In [ ]:
# AI에게 질문 보내기
response = client.responses.create(
    model=MODEL,
    input=prompt
)

# AI가 생성한 결과 가져오기
print(response.output_text)

In [ ]:
# AI 결과 생성에 사용한 비용 확인하기
usage = response.usage
print(usage)